In [ ]:
import joblib
import pandas as pd
import numpy as np
import sqlalchemy
import itertools
import json

# ⚙️ CONFIGURAÇÃO MESTRE DO PROJETO
MODO_SIMULACAO = 'estocastico' # 'estocastico' para zebras/emoção ou 'deterministico' para favoritismo puro

# Carga dos cérebros preditivos (Modelos Treinados na Sprint 3)
modelo = joblib.load('modelo_copa_2026.pkl')
scaler = joblib.load('scaler_copa_2026.pkl')

# Conexão segura com o nosso Data Lake local (SQL Server)
SERVER = 'FELIPE-PC\\SQLEXPRESS'
DATABASE = 'DB_COPA_2026'
connection_url = f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = sqlalchemy.create_engine(connection_url)

print(f"🚀 Ecossistema inicializado com sucesso! Modo ativo: [{MODO_SIMULACAO.upper()}]")

In [ ]:
# Query Sênior estruturada para ler metadados brutas da Staging e Core de forma performática
query_perfis = """
WITH CTE_Gols_Recentes AS (
    SELECT 
        ID_SELECAO,
        AVG(GOLS_MARCADOS) AS MED_GOLS_RECENTE
    FROM (
        SELECT ID_SELECAO_MANDANTE AS ID_SELECAO, GOLS_MANDANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
        UNION ALL
        SELECT ID_SELECAO_VISITANTE AS ID_SELECAO, GOLS_VISITANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
    ) AS todas_partidas
    WHERE DATA_PARTIDA >= DATEADD(MONTH, -24, (SELECT MAX(DATA_PARTIDA) FROM fato_partidas))
    GROUP BY ID_SELECAO
),
CTE_Ultimo_Ranking AS (
    SELECT 
        team,
        [total.points],
        ROW_NUMBER() OVER (PARTITION BY team ORDER BY date DESC) AS RN
    FROM stg_ranking_fifa
)
SELECT 
    s.ID_SELECAO,
    s.NOME_SELECAO,
    ISNULL(g.MED_GOLS_RECENTE, 0) AS MED_GOLS,
    ISNULL(r.[total.points], 1000) AS PONTOS_FIFA
FROM dim_selecoes s
LEFT JOIN CTE_Gols_Recentes g ON s.ID_SELECAO = g.ID_SELECAO
LEFT JOIN CTE_Ultimo_Ranking r ON s.NOME_SELECAO = r.team AND r.RN = 1;
"""

# Carrega e indexa a Lookup Table na memória RAM do Python
df_perfis_selecoes = pd.read_sql(query_perfis, engine)
df_perfis_selecoes.set_index('NOME_SELECAO', inplace=True)

# Consome a configuração estática dos grupos oficiais reais
with open('grupos_copa_2026.json', 'r', encoding='utf-8') as f:
    grupos_copa = json.load(f)

print(f"✅ Lookup Table ({df_perfis_selecoes.shape[0]} times) e Chaveamento Oficial carregados!")

✅ Lookup Table (364 times) e Chaveamento Oficial carregados!


In [ ]:
# 1. Criamos a query que traz TODOS os confrontos históricos de uma vez só, já com os nomes
query_cache_total = """
SELECT 
    m.NOME_SELECAO AS NOME_MANDANTE,
    v.NOME_SELECAO AS NOME_VISITANTE,
    vw.HISTORICO_CONFRONTOS_TOTAIS,
    vw.HISTORICO_VITORIAS_MANDANTE,
    vw.HISTORICO_VITORIAS_VISITANTE
FROM vw_confronto_direto vw
JOIN dim_selecoes m ON vw.ID_SELECAO_MANDANTE = m.ID_SELECAO
JOIN dim_selecoes v ON vw.ID_SELECAO_VISITANTE = v.ID_SELECAO
"""

# Carrega para a memória
df_bruto_h2h = pd.read_sql(query_cache_total, engine)

# Transforma num dicionário indexado por tuplas (Time_A, Time_B) para busca O(1) ultrarrápida
cache_h2h = {}
for _, row in df_bruto_h2h.iterrows():
    key = (row['NOME_MANDANTE'], row['NOME_VISITANTE'])
    cache_h2h[key] = {
        'HISTORICO_CONFRONTOS_TOTAIS': row['HISTORICO_CONFRONTOS_TOTAIS'],
        'HISTORICO_VITORIAS_MANDANTE': row['HISTORICO_VITORIAS_MANDANTE'],
        'HISTORICO_VITORIAS_VISITANTE': row['HISTORICO_VITORIAS_VISITANTE']
    }

# 2. Nova função buscar_historico_confronto blindada e sem tocar no banco de dados
def buscar_historico_confronto(nome_man, nome_vis):
    """
    Busca o histórico diretamente no cache mapeado em memória RAM.
    Zero conexões com o banco de dados durante o loop de simulação.
    """
    return cache_h2h.get((nome_man, nome_vis), {
        'HISTORICO_CONFRONTOS_TOTAIS': 0, 
        'HISTORICO_VITORIAS_MANDANTE': 0, 
        'HISTORICO_VITORIAS_VISITANTE': 0
    })

print(f"💾 Cache de confrontos diretos estruturado na memória RAM! ({len(cache_h2h)} registos prontos)")

⚙️ Funções auxiliares e preditores modulares integrados.


In [ ]:
classificacao_geral = {}

for nome_grupo, times in grupos_copa.items():
    tabela = pd.DataFrame(index=times, columns=['P', 'J', 'V', 'E', 'D', 'GP', 'GC', 'SG'])
    tabela.fillna(0, inplace=True)
    
    confrontos = list(itertools.combinations(times, 2))
    
    for time_man, time_vis in confrontos:
        res, p_man, p_vis = prever_jogo_unico(time_man, time_vis, modo=MODO_SIMULACAO)
        
        # Simulador de Gols e forças táticas
        if MODO_SIMULACAO == 'deterministico':
            gols_man_sim, gols_vis_sim = int(round(df_perfis_selecoes.loc[time_man, 'MED_GOLS'])), int(round(df_perfis_selecoes.loc[time_vis, 'MED_GOLS']))
        else:
            gols_man_sim, gols_vis_sim = int(np.random.poisson(df_perfis_selecoes.loc[time_man, 'MED_GOLS'])), int(np.random.poisson(df_perfis_selecoes.loc[time_vis, 'MED_GOLS']))
        
        if res == 2 and gols_man_sim <= gols_vis_sim: gols_man_sim = gols_vis_sim + 1
        elif res == 0 and gols_vis_sim <= gols_man_sim: gols_vis_sim = gols_man_sim + 1
        elif res == 1: gols_man_sim = gols_vis_sim = max(gols_man_sim, gols_vis_sim)
            
        tabela.loc[time_man, 'J'] += 1; tabela.loc[time_vis, 'J'] += 1
        tabela.loc[time_man, 'GP'] += gols_man_sim; tabela.loc[time_man, 'GC'] += gols_vis_sim
        tabela.loc[time_vis, 'GP'] += gols_vis_sim; tabela.loc[time_vis, 'GC'] += gols_man_sim
        
        if res == 2:
            tabela.loc[time_man, 'P'] += 3; tabela.loc[time_man, 'V'] += 1; tabela.loc[time_vis, 'D'] += 1
        elif res == 0:
            tabela.loc[time_vis, 'P'] += 3; tabela.loc[time_vis, 'V'] += 1; tabela.loc[time_man, 'D'] += 1
        else:
            tabela.loc[time_man, 'P'] += 1; tabela.loc[time_vis, 'P'] += 1
            tabela.loc[time_man, 'E'] += 1; tabela.loc[time_vis, 'E'] += 1
            
        tabela['SG'] = tabela['GP'] - tabela['GC']

    classificacao_geral[nome_grupo] = tabela.sort_values(by=['P', 'V', 'SG', 'GP'], ascending=False)

print("🏆 Fase de Grupos simulada com sucesso!")

🏆 Fase de Grupos simulada com sucesso!


In [ ]:
# Extração dos sobreviventes
primeiros, segundos, terceiros = {}, {}, []

for grupo, tabela in classificacao_geral.items():
    primeiros[grupo] = tabela.index[0]
    segundos[grupo] = tabela.index[1]
    
    dados_t = tabela.iloc[2].to_dict()
    dados_t['TIME'] = tabela.index[2]
    dados_t['GRUPO'] = grupo
    terceiros.append(dados_t)

# Ranking unificado para caçar os 8 melhores terceiros colocados
df_terceiros = pd.DataFrame(terceiros).sort_values(by=['P', 'V', 'SG', 'GP'], ascending=False).reset_index(drop=True)
melhores_3 = df_terceiros.head(8)['TIME'].tolist()

# 🗺️ MATRIZ DE CHAVEAMENTO OFICIAL DA FIFA (Round of 32 estruturado sem cruzamento repetido)
# Mapeamos os confrontos casando Líderes vs Segundos e os 8 terceiros sobreviventes (indexados de 0 a 7)
chaveamento_real_32 = [
    primeiros['Grupo A'], melhores_3[0],
    segundos['Grupo B'], segundos['Grupo C'],
    primeiros['Grupo E'], melhores_3[1],
    primeiros['Grupo F'], segundos['Grupo D'],
    primeiros['Grupo C'], melhores_3[2],
    segundos['Grupo A'], segundos['Grupo H'],
    primeiros['Grupo G'], melhores_3[3],
    primeiros['Grupo H'], segundos['Grupo F'],
    primeiros['Grupo I'], melhores_3[4],
    segundos['Grupo E'], segundos['Grupo L'],
    primeiros['Grupo K'], melhores_3[5],
    primeiros['Grupo L'], segundos['Grupo G'],
    primeiros['Grupo B'], melhores_3[6],
    segundos['Grupo J'], segundos['Grupo K'],
    primeiros['Grupo D'], melhores_3[7],
    primeiros['Grupo J'], segundos['Grupo I']
]

print(f"🚨 Árvore Realista de 32 times acoplada com sucesso!")

🚨 Árvore Realista de 32 times acoplada com sucesso!


In [ ]:
def simular_arvore_eliminatoria(lista_32_times, modo=MODO_SIMULACAO):
    fases = {
        "Round of 32": lista_32_times, "Oitavas de Final": [],
        "Quartas de Final": [], "Semifinal": [], "Grande Final": []
    }
    ordem = ["Round of 32", "Oitavas de Final", "Quartas de Final", "Semifinal"]
    
    for atual in ordem:
        times = fases[atual]
        proxima = list(fases.keys())[list(fases.keys()).index(atual) + 1]
        print(f"\n⚽ SIMULANDO: {atual.upper()}")
        
        for i in range(0, len(times), 2):
            t1, t2 = times[i], times[i+1]
            res, p1, p2 = prever_jogo_unico(t1, t2, modo=modo)
            
            if res == 1: # Empate em mata-mata força disputa de Pênaltis via Ranking FIFA
                vencedor = np.random.choice([t1, t2], p=[p1/(p1+p2), p2/(p1+p2)])
                print(f"  ⚖️ {t1} vs {t2} empataram. [{vencedor}] avançou nos pênaltis!")
            else:
                vencedor = t1 if res == 2 else t2
                print(f"  🟢 {vencedor} derrotou o adversário no tempo normal e avançou.")
            fases[proxima].append(vencedor)
            
    finais = fases["Grande Final"]
    print(f"\n🔥 🏟️ --- GRANDE FINAL DA COPA DO MUNDO: {finais[0].upper()} VS {finais[1].upper()} ---")
    c_res, pc1, pc2 = prever_jogo_unico(finais[0], finais[1], modo=modo)
    campeao = np.random.choice([finais[0], finais[1]], p=[pc1/(pc1+pc2), pc2/(pc1+pc2)]) if c_res == 1 else (finais[0] if c_res == 2 else finais[1])
    print(f"\n🏆 O [{campeao.upper()}] LEVANTA A TAÇA E É O CAMPEÃO DO MUNDO DE 2026!")
    return fases

# Executa o mata-mata completo integrado
historico_copa = simular_arvore_eliminatoria(chaveamento_real_32, modo=MODO_SIMULACAO)


⚽ SIMULANDO: ROUND OF 32
  🟢 Czech Republic derrotou o adversário no tempo normal e avançou.
  🟢 Canada derrotou o adversário no tempo normal e avançou.
  🟢 Germany derrotou o adversário no tempo normal e avançou.
  🟢 Paraguay derrotou o adversário no tempo normal e avançou.
  ⚖️ Morocco vs Tunisia empataram. [Tunisia] avançou nos pênaltis!
  🟢 South Africa derrotou o adversário no tempo normal e avançou.
  🟢 Belgium derrotou o adversário no tempo normal e avançou.
  🟢 Sweden derrotou o adversário no tempo normal e avançou.
  🟢 Senegal derrotou o adversário no tempo normal e avançou.
  🟢 Curaçao derrotou o adversário no tempo normal e avançou.
  🟢 New Zealand derrotou o adversário no tempo normal e avançou.
  🟢 Croatia derrotou o adversário no tempo normal e avançou.
  🟢 Qatar derrotou o adversário no tempo normal e avançou.
  ⚖️ Argentina vs Uzbekistan empataram. [Uzbekistan] avançou nos pênaltis!
  🟢 Bosnia and Herzegovina derrotou o adversário no tempo normal e avançou.
  ⚖️ Algeri

In [ ]:
def analisar_confronto_direto(time_man, time_vis, modo=MODO_SIMULACAO):
    """
    Exibe o raio-x completo do confronto: as probabilidades brutas da IA 
    e qual foi a decisão final do simulador de acordo com o modo ativo.
    """
    # 1. Recupera os dados e roda a IA
    perfis = df_perfis_selecoes.loc[[time_man, time_vis]]
    delta_ranking = perfis.loc[time_man, 'PONTOS_FIFA'] - perfis.loc[time_vis, 'PONTOS_FIFA']
    h2h = buscar_historico_confronto(time_man, time_vis)
    
    dados_confronto = pd.DataFrame([{
        'PESO_COMPETICAO': 3, 'DELTA_RANKING_PONTOS': delta_ranking,
        'MED_GOLS_MARCADOS_MANDANTE': perfis.loc[time_man, 'MED_GOLS'],
        'MED_GOLS_MARCADOS_VISITANTE': perfis.loc[time_vis, 'MED_GOLS'],
        'HISTORICO_CONFRONTOS_TOTAIS': h2h['HISTORICO_CONFRONTOS_TOTAIS'],
        'HISTORICO_VITORIAS_MANDANTE': h2h['HISTORICO_VITORIAS_MANDANTE'],
        'HISTORICO_VITORIAS_VISITANTE': h2h['HISTORICO_VITORIAS_VISITANTE']
    }])
    
    # 2. Captura as probabilidades brutas [Classe 0, Classe 1, Classe 2]
    probs = modelo.predict_proba(scaler.transform(dados_confronto))[0]
    p_vis, p_emp, p_man = probs[0] * 100, probs[1] * 100, probs[2] * 100
    
    # 3. Executa a decisão do modo
    if modo == 'deterministico':
        res = np.argmax(probs)
    else:
        res = np.random.choice([0, 1, 2], p=probs)
        
    # Mapeia o resultado para texto
    status_modelo = f"Vitória do {time_man}" if res == 2 else (f"Vitória do {time_vis}" if res == 0 else "Empate")
    
    # 4. Impressão do Painel Visual
    print(f"🔮 --- RAIO-X DA IA: {time_man.upper()} VS {time_vis.upper()} ---")
    print(f"📊 Probabilidade de Vitória do {time_man}: {p_man:.2f}%")
    print(f"🤝 Probabilidade de Empate: {p_emp:.2f}%")
    print(f"📊 Probabilidade de Vitória do {time_vis}: {p_vis:.2f}%")
    print(f"-" * 50)
    print(f"🎲 Modo Ativo: [{modo.upper()}]")
    print(f"🚨 Veredito do Simulador: {status_modelo.upper()}")
    print(f"=" * 50)

In [ ]:
analisar_confronto_direto('Brazil', 'Germany')

🔮 --- RAIO-X DA IA: BRAZIL VS GERMANY ---
📊 Probabilidade de Vitória do Brazil: 41.02%
🤝 Probabilidade de Empate: 25.69%
📊 Probabilidade de Vitória do Germany: 33.29%
--------------------------------------------------
🎲 Modo Ativo: [ESTOCASTICO]
🚨 Veredito do Simulador: VITÓRIA DO GERMANY
